# 03b. Baseline Characteristics by Early RAAS Exposure

Short summary:
[03b - Describe Analysis Dataset (Baseline Characteristics)](../docs/03b_describe_analysis_dataset_SHORT.md)

## 1. Purpose

This notebook summarizes baseline characteristics of non-ICU hospital admissions stratified by early RAAS inhibitor exposure using a pre-constructed, admission-level analysis dataset.

The goal is to describe cohort composition and baseline differences between exposure groups by summarizing demographic characteristics, comorbidity burden, and admission-related variables prior to outcome modeling.

This notebook is purely descriptive and does not fit outcome models, estimate effect measures, or perform causal inference.

### 1.1 Descriptive Scope

Admissions with documented early RAAS prescription records may differ systematically from those without such records with respect to demographic characteristics, comorbidity burden, and admission context.

Before estimating adjusted outcomes, it is therefore important to characterize baseline differences between exposure groups to understand potential sources of confounding and to contextualize subsequent multivariable analyses.

This notebook provides descriptive summaries only and does not attempt to estimate associations or causal effects.

## 2. Inputs

- **MIMIC-IV v3.1** (BigQuery public dataset)
- Project: `mimic-iv-portfolio`

**Source Tables:**
  - `mimic-iv-portfolio.nonicu_raas.analysis_dataset`<br>
    (created in [02_exposure.ipynb](02_exposure.ipynb) using [03_build_analysis_dataset.sql](../sql/03_build_analysis_dataset.sql))

These intermediate tables were generated using SQL-based preprocessing pipelines to ensure reproducibility and separation between data extraction and analysis.

## 3. Cohort Definition

### 3.1 Inclusion Criteria
- Adult admissions (age ≥ 18 years)
- Non-ICU hospital admissions
- Exposure indicators available at the admission level (with unexposed admissions explicitly coded)

Each row corresponds to a unique hospital admission.

## 4. Exposure Definition: Early RAAS Inhibitor Use

### 4.1 Exposure Construction
Early RAAS inhibitor exposure was defined upstream using pre-specified, time-restricted criteria and incorporated into the analysis dataset prior to this notebook.

For descriptive purposes, exposure status is represented using a binary admission-level indicator (`raas_any_early`), which indicates whether a qualifying ACE inhibitor or ARB prescription record has `admittime <= starttime < admittime + 24 hours`. It does not confirm actual administration, adherence or outpatient chronic use. Both-class flags do not establish simultaneous treatment.

Exposure definitions are not modified or re-derived in this notebook.

### 4.2 Exposure Group Labels
For descriptive clarity, exposure groups were labeled as:
- RAAS early
- No RAAS early

### Scope of these descriptive summaries

Each row is an admission; a patient can contribute multiple admissions. These tables describe the original entire-stay non-ICU association cohort, not the separate ward-confirmed admission/24-hour cohorts in [04c](04c_prediction_time_evaluation.ipynb). Hospital length of stay is a **whole-admission descriptive measure**, not an admission-time predictor. The analysis table was constructed in 02; 03a performs read-only checks. See [current execution evidence](../docs/CLINICAL_REEXECUTION_615de01.md).


## 5. Quality Checks

### 5.1 Common BigQuery Utilities

## Reading the results

Data sources and execution mode are summarized once below, after the destination settings. Expand **Calculation inputs and source definitions** to trace individual SQL definitions and aggregate references.

Each result is generated by Python in this order: **number/title → figure or aggregate table → how to read → computed result summary → interpretation limits**. Cohort definitions, exposures, adjustment variables and estimators are unchanged. See the [reproduction instructions](../REPRODUCIBILITY.md#python-generated-notebook-results).


The input-definition index describes the Python objects/queries used in calculation. The separate **Saved aggregates and verification files (CSV/JSON)** list identifies saved results and checking records, rather than original MIMIC-IV source tables. Live notebook displays use in-memory results; saved-output replay and validation workflows may read these files.

In [ ]:
# Presentation uses only objects produced by this notebook execution.
from pathlib import Path as _ResultPath
import sys as _result_sys
_result_root = _ResultPath.cwd().resolve()
if _result_root.name == "notebooks":
    _result_root = _result_root.parent
if not (_result_root / "scripts" / "nonicu_notebook_results.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or its notebooks directory")
if str(_result_root / "scripts") not in _result_sys.path:
    _result_sys.path.insert(0, str(_result_root / "scripts"))
from nonicu_notebook_results import NotebookResults
notebook_results = NotebookResults(globals())


In [ ]:
from google.cloud import bigquery
from google.auth import default
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Define project ID, dataset, and table references
PROJECT_ID = "mimic-iv-portfolio"
DATASET = "nonicu_raas"

TABLE_ANALYSIS = f"{PROJECT_ID}.{DATASET}.analysis_dataset"  # Created in 03_build_analysis_dataset.sql

# 2. Get ADC credentials

creds, adc_project = default()
client = bigquery.Client(project=PROJECT_ID, credentials=creds)

print("Connected to:", PROJECT_ID, "| ADC default:", adc_project)

In [ ]:
# Shared source references and execution mode for all numbered results.
notebook_results.overview(['Table 03b.1', 'Table 03b.2', 'Table 03b.3', 'Table 03b.4', 'Table 03b.5', 'Table 03b.6', 'Figure 03b.1'])


### 5.2 Dataset Loading and Initial Sanity Checks

In [ ]:
# 5. Helper for read-only SELECT queries → DataFrame
def query_to_df(query) :
    """
    Run a SELECT query in BigQuery and return a pandas DataFrame.
    """
    job = client.query(query)
    return job.to_dataframe(create_bqstorage_client=False)

In [ ]:
with notebook_results.capture('Table 03b.1', globals()):
    q = f"""
SELECT
  *
FROM `{TABLE_ANALYSIS}`
"""
    df = query_to_df(q)

    df.info()
    print("Shape:", df.shape)
    _notebook_result_value = (df["age"].describe())


### 5.3 Exposure Definition Validation and Consistency Checks

In [ ]:
with notebook_results.capture('Table 03b.2', globals()):
    # ================================
    # Basic dataset integrity checks
    # ================================

    # Number of rows (each row represents one hospital admission)
    print("Rows:", len(df))

    # Number of unique hospital admissions (should match row count)
    print("Unique hadm_id:", df["hadm_id"].nunique())


    # ================================
    # Exposure prevalence
    # ================================

    # Proportion of admissions with early RAAS inhibitor exposure
    # Since raas_any_early is coded as 0/1, the mean corresponds to prevalence
    prev = df["raas_any_early"].mean()
    print(f"Early RAAS exposure prevalence: {prev:.4f} ({prev*100:.2f}%)")


    # ================================
    # Internal consistency checks for exposure definitions
    # ================================

    # Check that raas_any_early is correctly defined as:
    # 1 if either ACE inhibitor OR ARB has a qualifying early prescription record
    raas_any_definition_check = (
        ((df["acei_early"] == 1) | (df["arb_early"] == 1))
        == (df["raas_any_early"] == 1)
    )

    # Check that raas_both_early is correctly defined as:
    # 1 if BOTH classes have early prescription records; no administration/overlap requirement
    raas_both_definition_check = (
        ((df["acei_early"] == 1) & (df["arb_early"] == 1))
        == (df["raas_both_early"] == 1)
    )

    # Proportion of rows where definitions are internally consistent
    # A value of 1.00 indicates perfect agreement across all admissions
    print(
        "RAAS-any indicator matches ACEi or ARB exposure:",
        f"{raas_any_definition_check.mean():.2f}"
    )

    _notebook_result_value = (print(
        "RAAS-both indicator matches both early prescription-record flags (not simultaneous administration):",
        f"{raas_both_definition_check.mean():.2f}"
    ))


### 5.4 Exposure Group Label Construction for Descriptive Analysis

In [ ]:
df["expo_group"] = np.where(df["raas_any_early"] == 1, "RAAS early", "No RAAS early")

In [ ]:
with notebook_results.capture('Table 03b.3', globals()):
    # Exposure group counts with percentages
    expo_summary = (
        df["expo_group"]
        .value_counts(dropna=False)
        .rename("n")
        .to_frame()
    )

    expo_summary["pct"] = (expo_summary["n"] / expo_summary["n"].sum() * 100).round(2)

    _notebook_result_value = (expo_summary.reset_index().rename(
        columns={
            "index": "Exposure group",
            "n": "Number of admissions",
            "pct": "Percentage (%)"
        }
    ))


### 5.5 Baseline Continuous Variables by Exposure Group

In [ ]:
with notebook_results.capture('Table 03b.4', globals()):
    # ------------------------------------------------------------
    # Helper functions for formatted descriptive statistics
    # ------------------------------------------------------------

    def format_mean_sd(x):
        """
    Return mean and standard deviation formatted as:
    mean (sd)
    """
        x = x.dropna()
        if len(x) == 0:
            return "NA"
        return f"{x.mean():.2f} ({x.std():.2f})"


    def format_median_iqr(x):
        """
    Return median and interquartile range formatted as:
    median [Q1, Q3]
    """
        x = x.dropna()
        if len(x) == 0:
            return "NA"
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        return f"{x.median():.2f} [{q1:.2f}, {q3:.2f}]"


    # ------------------------------------------------------------
    # Continuous variables to summarize
    # ------------------------------------------------------------
    cont_vars = ["age", "hosp_los"]


    # ------------------------------------------------------------
    # Build Table 1: Continuous baseline characteristics
    # Stratified by early RAAS exposure
    # ------------------------------------------------------------
    # ------------------------------------------------------------
    # Helper functions for formatted descriptive statistics
    # ------------------------------------------------------------
    def format_mean_sd(x):
        x = x.dropna()
        if len(x) == 0:
            return "NA"
        return f"{x.mean():.2f} ({x.std():.2f})"

    def format_median_iqr(x):
        x = x.dropna()
        if len(x) == 0:
            return "NA"
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        return f"{x.median():.2f} [{q1:.2f}, {q3:.2f}]"

    # ------------------------------------------------------------
    # Continuous variables to summarize
    # ------------------------------------------------------------
    cont_vars = ["age", "hosp_los"]

    # ------------------------------------------------------------
    # Build Table 1: Continuous baseline characteristics
    # Stratified by early RAAS exposure (raas_any_early)
    # ------------------------------------------------------------
    rows = []

    for v in cont_vars:
        g0 = df.loc[df["raas_any_early"] == 0, v]
        g1 = df.loc[df["raas_any_early"] == 1, v]

        rows.append({
            "variable": v,
            "No RAAS early (mean (sd))": format_mean_sd(g0),
            "RAAS early (mean (sd))": format_mean_sd(g1),
            "No RAAS early (median [IQR])": format_median_iqr(g0),
            "RAAS early (median [IQR])": format_median_iqr(g1),
        })

    table1_cont = pd.DataFrame(rows)
    _notebook_result_value = (table1_cont)


### 5.6 Baseline Categorical Variables by Exposure Group

In [ ]:
def cat_table_wide(df: pd.DataFrame, var: str, group_col: str = "expo_group") -> pd.DataFrame:
    # Count
    ct = pd.crosstab(df[var], df[group_col], dropna=False)

    # Percent within each exposure group (column-wise)
    pct = ct.div(ct.sum(axis=0), axis=1) * 100

    # Combine as "n (pct%)"
    out = pd.DataFrame(index=ct.index)
    for g in ct.columns:
        out[g] = ct[g].astype(int).astype(str) + " (" + pct[g].round(2).astype(str) + "%)"

    # Optional: add percentage-point difference (RAAS - No RAAS) if both exist
    if ("RAAS early" in ct.columns) and ("No RAAS early" in ct.columns):
        out["pp_diff (RAAS - No RAAS)"] = (pct["RAAS early"] - pct["No RAAS early"]).round(2)

    out = out.reset_index().rename(columns={var: "category"})
    return out

In [ ]:
with notebook_results.capture('Table 03b.5', globals()):
    _notebook_result_value = (cat_table_wide(df, "gender"))


Baseline categorical characteristics by early RAAS inhibitor exposure

Values are shown as *n (% within exposure group)*.  
**pp_diff** denotes the absolute difference in percentage points between the RAAS early group and the No RAAS early group (RAAS − No RAAS).

### 5.7 Unadjusted Baseline Summary by Early RAAS Inhibitor Exposure

In [ ]:
with notebook_results.capture('Table 03b.6', globals()):
    summary = df.groupby("expo_group").agg(
        n=("hadm_id", "count"),
        age_mean=("age", "mean"),
        age_sd=("age", "std"),
        los_median=("hosp_los", "median"),
        raas_any=("raas_any_early", "mean"),
        acei=("acei_early", "mean"),
        arb=("arb_early", "mean"),
        both=("raas_both_early", "mean"),
    )
    _notebook_result_value = (summary)


Baseline summary statistics by early RAAS inhibitor exposure

Values are reported by exposure group. Continuous variables are shown as means (SD) or medians, as appropriate. Binary variables represent proportions within each group.

## 6. Outputs

This notebook generates baseline descriptive summaries stratified by early RAAS inhibitor exposure and saves them as reusable interim artifacts for downstream analyses.　These outputs are used as baseline descriptive inputs (Table 1–style summaries) for adjusted outcome modeling and marginal effect estimation in subsequent notebooks ([04a](04a_unadjusted_outcomes.ipynb), [04b](04b_multivariable_outcomes.ipynb)).

The following outputs are produced:

- **Baseline continuous-variable summary table (table1_cont.csv)**
	- Includes age and hospital length of stay summarized as mean (SD) and median [IQR]
	- Stratified by early RAAS exposure group
	- Used for baseline characterization and reporting

- **Group-level baseline summary table (baseline_summary.csv)**
	- Includes sample size, age distribution, length of stay, and exposure composition
	- Serves as a compact reference for interpreting adjusted analyses

- **On-screen categorical distribution tables**
	- Sex, admission type, and calendar period distributions by exposure group
	- Used for exploratory checks and cohort understanding (not exported)

### 6.1 Baseline Summary Table Export


In [ ]:
from pathlib import Path

out_path = Path("../data/interim")
out_path.mkdir(parents=True, exist_ok=True)

table1_cont.to_csv(out_path / "table1_cont.csv", index=False)
summary.to_csv(out_path / "baseline_summary.csv")

print("Saved to:", out_path)

### 6.1 Portfolio Figure Export: Baseline Characteristics

This export uses the existing baseline summary objects and does not change the descriptive analysis.


In [ ]:
with notebook_results.capture('Figure 03b.1', globals()):
    # Export portfolio figure: baseline characteristics summary
    import matplotlib.pyplot as plt

    notebooks_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
    project_root = notebooks_dir.parent
    assets_dir = project_root / "assets"
    assets_dir.mkdir(parents=True, exist_ok=True)
    figure_path = assets_dir / "baseline_characteristics_summary.png"

    plot_df = summary.copy()
    plot_df["group"] = plot_df.index.astype(str)
    plot_df = plot_df.loc[["No RAAS early", "RAAS early"]]

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
    colors = ["#64748b", "#2563eb"]

    axes[0].bar(plot_df["group"], plot_df["n"] / 1000, color=colors)
    axes[0].set_ylabel("Admissions (thousands)")
    axes[0].set_title("Cohort size")

    axes[1].bar(plot_df["group"], plot_df["age_mean"], color=colors)
    axes[1].set_ylabel("Mean age (years)")
    axes[1].set_title("Age")

    axes[2].bar(plot_df["group"], plot_df["los_median"], color=colors)
    axes[2].set_ylabel("Median LOS (days)")
    axes[2].set_title("Hospital LOS")

    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(axis="x", rotation=20)

    fig.suptitle("Baseline characteristics by early RAAS exposure", y=1.04)
    fig.tight_layout()
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()

    _notebook_result_value = (figure_path.relative_to(project_root))


---
## Next Step

Proceed to:
- [04a. Unadjusted Outcomes Overview](04a_unadjusted_outcomes.ipynb)

This next notebook/document briefly describes the next workflow step.